# 7.3. Cross-Validation & Evaluation Strategies

**Learning Objectives:**
- Understand why naive train/test splits can be misleading for brain decoding
- Compare different **cross-validation (CV)** strategies (K-fold, stratified, group-based)
- Learn how to avoid **data leakage** in neuroimaging analyses
- Implement **nested cross-validation** for robust hyperparameter tuning
- Use **permutation testing** to assess statistical significance of decoding results

**Key Concepts:**
- **Train/Test Split**: One-off split of data into training and test sets
- **K-Fold CV**: Repeatedly train/test on different splits for more stable estimates
- **Stratified CV**: Preserve class balance in each fold (important for imbalanced labels)
- **Group CV**: Ensure that all samples from a group (e.g., subject/run) are in the same fold
- **Nested CV**: Inner loop for model selection, outer loop for unbiased performance estimation
- **Data Leakage**: Information from the test set leaking into training (e.g., scaling, feature selection, or grouping mistakes)
- **Permutation Testing**: Building a null distribution by shuffling labels

In notebooks **7.1** and **7.2**, we applied these ideas to real EEG and fMRI data. Here, we'll use **synthetic datasets** so we can quickly and clearly explore different CV strategies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    GroupKFold,
    LeaveOneGroupOut,
    cross_val_score,
    GridSearchCV,
    permutation_test_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')

sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

rng = np.random.RandomState(42)

## 1. A Simple Synthetic Decoding Problem

We'll start with a **toy classification task** that mimics decoding conditions from brain features:
- 2 classes (e.g., condition A vs B)
- 20 features (e.g., sensors/voxels)
- Some informative, some noisy

We'll compare:
- **Single train/test split**
- **K-fold cross-validation**

Using a **logistic regression** classifier:

In [ ]:
# Create a synthetic dataset
X, y = make_classification(
    n_samples=300,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.05,
    random_state=42,
)

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class balance: {np.bincount(y)}")

# Define a simple pipeline: scaling + logistic regression
clf_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='liblinear', random_state=42)
)
print(clf_lr)

In [ ]:
# 1.1 Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf_lr.fit(X_train, y_train)
acc_train = clf_lr.score(X_train, y_train)
acc_test = clf_lr.score(X_test, y_test)

print(f"Train accuracy (single split): {acc_train:.3f}")
print(f"Test accuracy  (single split): {acc_test:.3f}")

In [ ]:
# 1.2 K-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf_lr, X, y, cv=cv, n_jobs=1)

print("Cross-validated accuracy (Stratified 5-fold):")
for i, s in enumerate(scores, 1):
    print(f"  Fold {i}: {s:.3f}")
print(f"\nMean accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

### Takeaway

- **Train/test split** gives one estimate, which can fluctuate depending on the random split.
- **K-fold CV** averages performance across many splits, giving a more **stable and reliable** estimate.

In real EEG/fMRI decoding (7.1, 7.2), we almost always prefer **cross-validation** over a single split.

## 2. Stratified vs Non-Stratified CV

For **imbalanced labels**, we want each fold to preserve the class proportions.

We'll create a slightly imbalanced dataset and compare:
- Plain **KFold** (may create folds with skewed class balance)
- **StratifiedKFold** (preserves class balance in each fold)

In [ ]:
# Create an imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=400,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    weights=[0.75, 0.25],  # 3:1 class imbalance
    random_state=42,
)

print("Class balance (imbalanced):", np.bincount(y_imb))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def fold_class_counts(cv, X, y):
    counts = []
    for _, idx_test in cv.split(X, y):
        counts.append(np.bincount(y[idx_test], minlength=2))
    return np.array(counts)

counts_kf = fold_class_counts(kf, X_imb, y_imb)
counts_skf = fold_class_counts(skf, X_imb, y_imb)

print("\nKFold class counts per fold (test):\n", counts_kf)
print("\nStratifiedKFold class counts per fold (test):\n", counts_skf)

### Takeaway

- **StratifiedKFold** keeps the label distribution similar across folds.
- This is essential for **imbalanced decoding problems**, so each fold has enough trials of each class.
- In 7.1/7.2 we used **stratified CV** by default.

## 3. Group / Run-Based CV (Avoiding Leakage)

In neuroimaging, we often have **groups** of samples that must be kept together:
- **Runs** within a subject (e.g., fMRI runs)
- **Subjects** in multi-subject analyses

If we mix runs/subjects across train and test folds, we can get artificially high accuracy due to **temporal autocorrelation** or **subject-specific patterns**.

We'll simulate this by:
- Creating synthetic data with **run-specific shifts**
- Comparing:
  - StratifiedKFold (leaky: runs can be split across train/test)
  - GroupKFold / LeaveOneGroupOut (correct: runs kept separate)

In [ ]:
# Simulate "runs" with run-specific shifts
n_runs = 8
n_per_run = 60
n_samples = n_runs * n_per_run

X_base, y_base = make_classification(
    n_samples=n_samples,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    random_state=123,
)

# Assign run labels (groups)
runs = np.repeat(np.arange(n_runs), n_per_run)

# Add run-specific offsets to features (simulating run-wise noise/drift)
run_offsets = rng.normal(scale=2.0, size=(n_runs, 1))  # large run-specific shifts
X_runs = X_base + run_offsets[runs]

print("X shape:", X_runs.shape)
print("y shape:", y_base.shape)
print("Runs:", np.unique(runs))

In [ ]:
clf_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='linear', C=1.0, random_state=42)
)

# 3.1 Leaky CV: stratified over individual samples (ignores runs)
cv_leaky = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_leaky = cross_val_score(clf_svm, X_runs, y_base, cv=cv_leaky, n_jobs=1)

# 3.2 Correct CV: group-based (runs kept together)
cv_group = GroupKFold(n_splits=n_runs)  # or LeaveOneGroupOut()
scores_group = cross_val_score(clf_svm, X_runs, y_base, cv=cv_group.split(X_runs, y_base, groups=runs), n_jobs=1)

print("Leaky CV (StratifiedKFold over trials):")
print("  Fold scores:", np.round(scores_leaky, 3))
print(f"  Mean accuracy: {scores_leaky.mean():.3f} ± {scores_leaky.std():.3f}\n")

print("Group CV (GroupKFold over runs):")
print("  Fold scores:", np.round(scores_group, 3))
print(f"  Mean accuracy: {scores_group.mean():.3f} ± {scores_group.std():.3f}")

### Takeaway

- The **leaky CV** (ignoring groups) can produce **inflated accuracy** because the model sees very similar run-specific patterns in both train and test.
- **Group-based CV** (e.g., `GroupKFold`, `LeaveOneGroupOut`) is crucial when you have **runs** or **subjects**.
- In 7.2, we used `LeaveOneGroupOut` over fMRI runs (`chunks`) to avoid leakage.

In real neuroimaging analyses, always ask:
- *"What are my groups (runs/subjects)?"*
- *"Could information leak across them if I split trials randomly?"*

## 4. Nested Cross-Validation for Hyperparameter Tuning

If we use the **same data** for both:
- Choosing hyperparameters (e.g., `C` for SVM)
- Estimating final accuracy

we can **overfit to the validation folds**.

+The solution is **nested cross-validation**:
- **Inner loop**: model selection / hyperparameter tuning
- **Outer loop**: unbiased performance estimate

We'll demonstrate nested CV on our synthetic decoding problem.

In [ ]:
# Use the original (non-run) dataset X, y
X_nested, y_nested = X, y

# Define a pipeline and parameter grid for SVM
pipe_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='linear', random_state=42)
)

param_grid = {
    'svc__C': [0.01, 0.1, 1.0, 10.0]
}

# Inner CV for hyperparameter tuning
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid,
    cv=inner_cv,
    scoring='accuracy',
    n_jobs=1,
)

# Outer CV for performance estimation
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nested_scores = cross_val_score(grid, X_nested, y_nested, cv=outer_cv, n_jobs=1)

print("Nested CV accuracy (outer folds):")
for i, s in enumerate(nested_scores, 1):
    print(f"  Fold {i}: {s:.3f}")
print(f"\nMean nested CV accuracy: {nested_scores.mean():.3f} ± {nested_scores.std():.3f}")

### Takeaway

- **Nested CV** separates **model selection** from **performance estimation**.
- This is particularly important when you have **many hyperparameters** or **complex pipelines**.
- It is more computationally expensive, but gives a more **honest accuracy estimate**.

In practice, you might:
- Use nested CV for **final benchmarking**
- Use a simpler CV (with careful validation) during rapid prototyping

## 5. Permutation Testing for Statistical Significance

Cross-validation gives an **accuracy estimate**, but not whether that accuracy is **statistically above chance**.

Permutation testing (as in 7.1 and 7.2):
1. Shuffle labels (destroy true relationship)
2. Re-run CV decoding
3. Repeat many times to build a **null distribution** of accuracies
4. Compute **p-value**: fraction of permutations with accuracy ≥ true accuracy


In [ ]:
# Use the simple SVM pipeline and stratified CV
cv_perm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running permutation test (50 permutations)...")
score_true, perm_scores, pvalue = permutation_test_score(
    pipe_svm,
    X_nested,
    y_nested,
    scoring='accuracy',
    cv=cv_perm,
    n_permutations=50,
    n_jobs=1,
    random_state=42,
)

print(f"True accuracy: {score_true:.3f}")
print(f"Permutation mean: {perm_scores.mean():.3f}")
print(f"Permutation std: {perm_scores.std():.3f}")
print(f"P-value: {pvalue:.4f}")

In [ ]:
# Visualize permutation distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(perm_scores, bins=15, color='gray', alpha=0.7, edgecolor='black',
        label='Null distribution (permuted labels)')

ax.axvline(score_true, color='red', linestyle='--', linewidth=3,
           label=f'True accuracy = {score_true:.3f}\np = {pvalue:.4f}')
ax.axvline(0.5, color='blue', linestyle=':', linewidth=2, label='Chance (0.5)')

ax.set_xlabel('Accuracy')
ax.set_ylabel('Count')
ax.set_title('Permutation Test: Null Distribution vs True Accuracy')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Takeaway

- Permutation testing tells us **how surprising** our accuracy is under the null hypothesis (no relationship between data and labels).
- A small **p-value** (e.g., < 0.05) suggests that decoding is **significantly above chance**.
- In 7.1 and 7.2, we used permutation tests on real EEG/fMRI decoding results.


## 6. Summary & Best Practices

**For neuroimaging decoding (EEG/MEG/fMRI):**

- **Always use cross-validation** rather than a single train/test split.
- Prefer **StratifiedKFold** for balanced label distributions, especially with imbalanced classes.
- Use **GroupKFold** or **LeaveOneGroupOut** when you have **runs or subjects** to avoid leakage.
- For hyperparameter tuning, use **nested CV** to avoid optimistic bias.
- Use **permutation testing** to assess whether decoding is **statistically above chance**.

In the next tutorials, these principles will carry over to **deep learning** (08.x) and **multimodal decoding** (09.x), where CV design remains just as critical as the model architecture.